In [2]:
import pandas as pd
df = pd.read_csv("tiktok_NousToutes_ALL.csv")
df.head(5)


,Unnamed: 0,videoMeta.coverUrl,text,diggCount,shareCount,playCount,commentCount,videoMeta.duration,locationCreated,isAd,...,hashtags/8/name,hashtags/9/name,hashtags/10/name,hashtags/11/name,hashtags/12/name,hashtags/13/name,hashtags/14/name,authorMeta.name,webVideoUrl,createTimeISO
0,0,https://p19-common-sign.tiktokcdn-us.com/tos-u...,noustoutes n est pas une association qui peut ...,544,16,6263,36,102,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,noustoutesorg,https://www.tiktok.com/@noustoutesorg/video/73...,2024-03-12T15:45:56.000Z
1,1,https://p16-common-sign.tiktokcdn-us.com/tos-u...,contre les volences de genre novembre feminism...,44100,111,197300,103,0,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,noustoutesorg,https://www.tiktok.com/@noustoutesorg/video/73...,2023-11-28T13:14:37.000Z
2,2,https://p16-common-sign.tiktokcdn-us.com/tos-u...,retrouvez les benevoles de noustoutes et de ha...,16000,523,122500,66,29,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,noustoutesorg,https://www.tiktok.com/@noustoutesorg/video/72...,2023-08-27T13:50:24.000Z
3,3,https://p16-common-sign.tiktokcdn-us.com/tos-n...,soutien infini aux victimes de vilances mepris...,238,7,4211,8,0,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,noustoutesorg,https://www.tiktok.com/@noustoutesorg/video/75...,2025-12-08T20:34:43.000Z
4,4,https://p19-common-sign.tiktokcdn-us.com/tos-n...,brigitte macron parle de militantes de noustou...,331,38,3686,18,20,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,noustoutesorg,https://www.tiktok.com/@noustoutesorg/video/75...,2025-12-08T20:21:11.000Z


In [3]:
#drop lines with no text (NaN)
df_clean = df.dropna()

In [4]:
pip install wordcloud

Note: you may need to restart the kernel to use updated packages.


In [5]:
import re
import unicodedata
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet, stopwords
from nltk.stem import WordNetLemmatizer


In [6]:
#FIRST STEP: we remove accents from all text in the message column, as well as special characters and emojis...

def remove_special_characters(text):
    if not isinstance(text, str):
        return text
    
    # normalize the text (breaks down "é" into "e" + "combining accent")
    normalized = unicodedata.normalize('NFD', text)
    
    # strip out accents
    text = "".join(c for c in normalized if unicodedata.category(c) != 'Mn')

    # replace apostrophes by a blank space
    text = text.replace("'", " ").replace("’", " ")
    
    # apply existing regex to remove remaining non-alphabetic characters
    return re.sub(r"[^A-Za-z\s]", "", text)

df["text"] = df["text"].apply(remove_special_characters)
df["text"] = df["text"].str.lower()
df.head()

,Unnamed: 0,videoMeta.coverUrl,text,diggCount,shareCount,playCount,commentCount,videoMeta.duration,locationCreated,isAd,...,hashtags/8/name,hashtags/9/name,hashtags/10/name,hashtags/11/name,hashtags/12/name,hashtags/13/name,hashtags/14/name,authorMeta.name,webVideoUrl,createTimeISO
0,0,https://p19-common-sign.tiktokcdn-us.com/tos-u...,noustoutes n est pas une association qui peut ...,544,16,6263,36,102,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,noustoutesorg,https://www.tiktok.com/@noustoutesorg/video/73...,2024-03-12T15:45:56.000Z
1,1,https://p16-common-sign.tiktokcdn-us.com/tos-u...,contre les volences de genre novembre feminism...,44100,111,197300,103,0,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,noustoutesorg,https://www.tiktok.com/@noustoutesorg/video/73...,2023-11-28T13:14:37.000Z
2,2,https://p16-common-sign.tiktokcdn-us.com/tos-u...,retrouvez les benevoles de noustoutes et de ha...,16000,523,122500,66,29,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,noustoutesorg,https://www.tiktok.com/@noustoutesorg/video/72...,2023-08-27T13:50:24.000Z
3,3,https://p16-common-sign.tiktokcdn-us.com/tos-n...,soutien infini aux victimes de vilances mepris...,238,7,4211,8,0,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,noustoutesorg,https://www.tiktok.com/@noustoutesorg/video/75...,2025-12-08T20:34:43.000Z
4,4,https://p19-common-sign.tiktokcdn-us.com/tos-n...,brigitte macron parle de militantes de noustou...,331,38,3686,18,20,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,noustoutesorg,https://www.tiktok.com/@noustoutesorg/video/75...,2025-12-08T20:21:11.000Z


In [7]:
df['text'] = df['text'].str.replace(r'(\\n)+', ' ', regex=True)

# replace actual line breaks AND literal '\n' strings with a space
df['text'] = df['text'].str.replace(r'(\\n|\n|\r)+', ' ', regex=True)

# collapse any multiple spaces created into a single space
df['text'] = df['text'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [8]:
df['text']

0      noustoutes n est pas une association qui peut ...
1      contre les volences de genre novembre feminism...
2      retrouvez les benevoles de noustoutes et de ha...
3      soutien infini aux victimes de vilances mepris...
4      brigitte macron parle de militantes de noustou...
                             ...                        
200    besoin de ricaner un instant pour oublier que ...
201    er challenge dis moi que tu es feministe je su...
202    noustoutes organisera une marche a paris le po...
203    rappel les femmes sont libres natalie wood sx ...
204    noustoutes debarque sur tiktok qui a dit que l...
Name: text, Length: 205, dtype: str

In [9]:
df["words"] = df["text"].str.split()
df.head()

,Unnamed: 0,videoMeta.coverUrl,text,diggCount,shareCount,playCount,commentCount,videoMeta.duration,locationCreated,isAd,...,hashtags/9/name,hashtags/10/name,hashtags/11/name,hashtags/12/name,hashtags/13/name,hashtags/14/name,authorMeta.name,webVideoUrl,createTimeISO,words
0,0,https://p19-common-sign.tiktokcdn-us.com/tos-u...,noustoutes n est pas une association qui peut ...,544,16,6263,36,102,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,noustoutesorg,https://www.tiktok.com/@noustoutesorg/video/73...,2024-03-12T15:45:56.000Z,"[noustoutes, n, est, pas, une, association, qu..."
1,1,https://p16-common-sign.tiktokcdn-us.com/tos-u...,contre les volences de genre novembre feminism...,44100,111,197300,103,0,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,noustoutesorg,https://www.tiktok.com/@noustoutesorg/video/73...,2023-11-28T13:14:37.000Z,"[contre, les, volences, de, genre, novembre, f..."
2,2,https://p16-common-sign.tiktokcdn-us.com/tos-u...,retrouvez les benevoles de noustoutes et de ha...,16000,523,122500,66,29,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,noustoutesorg,https://www.tiktok.com/@noustoutesorg/video/72...,2023-08-27T13:50:24.000Z,"[retrouvez, les, benevoles, de, noustoutes, et..."
3,3,https://p16-common-sign.tiktokcdn-us.com/tos-n...,soutien infini aux victimes de vilances mepris...,238,7,4211,8,0,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,noustoutesorg,https://www.tiktok.com/@noustoutesorg/video/75...,2025-12-08T20:34:43.000Z,"[soutien, infini, aux, victimes, de, vilances,..."
4,4,https://p19-common-sign.tiktokcdn-us.com/tos-n...,brigitte macron parle de militantes de noustou...,331,38,3686,18,20,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,noustoutesorg,https://www.tiktok.com/@noustoutesorg/video/75...,2025-12-08T20:21:11.000Z,"[brigitte, macron, parle, de, militantes, de, ..."


In [10]:
# convert everything to a string 
all_words2 = " ".join(
    [" ".join(str(w) for w in words) for words in df["words"].dropna()]
)

In [11]:
import spacy

nlp = spacy.load("fr_core_news_sm")

# define base lemmas of French auxiliary and modal verbs to exclude
AUXILIARY_LEMMAS = {
    "être",
    "avoir",
    "faire",
    "devoir",
    "pouvoir",
    "vouloir",
    "falloir",
}

EXCLUDE_POS = {"AUX", "DET", "ADP", "CCONJ", "SCONJ", "PUNCT", "PRON"}

doc = nlp(all_words2)

filtered_words = [
    token.text
    for token in doc
    if token.pos_ not in EXCLUDE_POS
    and token.lemma_.lower() not in AUXILIARY_LEMMAS
]

In [12]:
from collections import Counter

In [13]:
# define the words you want to remove
words_to_remove = {
    "n",
    "pas",
    "c",
    "ne",
    "h",
    "etes",
    "ete",
    "bla",
    "ca"
}

# filter
filtered_words2 = [word for word in filtered_words if word not in words_to_remove]

In [14]:
# remove word "plus"
cleaned_words3 = [word for word in filtered_words2 if word != "plus"]

In [15]:
counts2 = Counter(cleaned_words3)
counts2

Counter({'noustoutes': 75,
         'femmes': 47,
         'violences': 44,
         'victimes': 34,
         'france': 33,
         'toutes': 28,
         'novembre': 25,
         'partout': 19,
         'vss': 19,
         'feminisme': 18,
         'sexuelles': 17,
         'personnes': 16,
         'macron': 16,
         'sexistes': 14,
         'dire': 14,
         'merci': 14,
         'paris': 14,
         'extreme': 13,
         'droite': 13,
         'vilences': 13,
         'feministe': 12,
         'droits': 11,
         'feministes': 11,
         'croit': 11,
         'justice': 11,
         'infos': 10,
         'soutien': 10,
         'rdv': 10,
         'stop': 10,
         'faites': 9,
         'samedi': 9,
         'noustoutesorg': 9,
         'depardieu': 9,
         'genre': 8,
         'non': 8,
         'place': 8,
         'mars': 8,
         'annee': 8,
         'tout': 8,
         'rue': 8,
         'hommes': 8,
         'gouvernement': 8,
         'besoin': 7,
 

In [16]:
number_all_words = len(cleaned_words3)

In [17]:
#Look at the highest occurrences of words

df1 = pd.DataFrame(counts2.items(), columns=["Word", "Count"])

# calculate percentage
df1["Percentage"] = (df1["Count"] / number_all_words) * 100

# sort by highest percentage/count
df1 = df1.sort_values(by="Percentage", ascending=False).reset_index(drop=True)

# format the percentage column 
df1["Percentage"] = df1["Percentage"].map("{:.2f}%".format)

# display Top 20
df1.head(20)

,Word,Count,Percentage
0,noustoutes,75,2.54%
1,femmes,47,1.59%
2,violences,44,1.49%
3,victimes,34,1.15%
4,france,33,1.12%
5,toutes,28,0.95%
6,novembre,25,0.85%
7,vss,19,0.64%
8,partout,19,0.64%
9,feminisme,18,0.61%


In [26]:
# 1. Define the GBV French Lexicon
GBV_FRENCH_LEXICON = {
    # Core violences
    "sodomie",
    "pénétration",
    "fellation",
    "agression",
    "abus",
    "meurtre",
    "meurtri",
    "tuer",
    "assassinat",
    "barbarie",
    "torture",
    "supplice",
    # Physical acts
    "coup",
    "frapper",
    "tabasser",
    "lyncher",
    "rouer",
    "poing",
    "claque",
    "gifler",
    "étrangler",
    "strangulation",
    "étouffer",
    "ligoter",
    "attacher",
    "suspendre",
    "mordre",
    "mordue",
    "tirer",
    "pousser",
    "exhibition",
    # Physical state & weapons
    "hématome",
    "contusion",
    "dermabrasion",
    "tuméfié",
    "gonflé",
    "déformé",
    "blessé",
    "ensanglanté",
    "sang",
    "couteau",
    "lame",
    "arme",
    "calibre",
    "corps",
    "nu",
    "nue",
    "déshabiller",
    "incapacité",
    # Predatory & Exploitation
    "proie",
    "repérer",
    "suivre",
    "traquer",
    "menace",
    "punition",
    "bourreau",
    "filmer",
    "publier",
    "darkweb",
    "film",
    "vidéo",
    "bande",
    "criminel",
    "faveur",
    # Slurs & Dehumanization
    "pute",
    "salope",
    "chienne",
    "rut",
    "baise",
    "niquer",
    "insulte",
    "consommer",
    "animal",
    "sacrifier",
    "vice",
    # Emotional/Qualitative
    "horreur",
    "choquant",
    "choc",
    "effroyable",
    "humilié",
    "humiliation",
    "drame",
    "insoutenable",
    "sordide",
    "abominable",
    "brutalement",
    "hurler",
    "hurlant",
    "irréparable",
    "sacrifié",
}

words = cleaned_words3  

target_phrases = ["violence"]
target_words = set(target_phrases)

window_size = 20
co_terms = Counter()
gbv_counts = Counter()  # <--- NEW: Dedicated counter for GBV words
examples = []
matches = 0

for i in range(len(words)):
    phrase = words[i]

    if phrase in target_phrases:
        matches += 1

        start = max(0, i - window_size)
        end = min(len(words), i + 1 + window_size)
        context_words = words[start:end]

        # Save context window examples (up to 5)
        if len(examples) < 5:
            left = " ".join(words[max(0, i - window_size) : i])
            right = " ".join(
                words[i + 1 : min(len(words), i + 1 + window_size)]
            )
            examples.append(f"{left} [{phrase.upper()}] {right}")

        # Count co-occurring terms
        for word in context_words:
            if word.isalpha() and word not in target_words and len(word) > 2:
                # Track overall co-occurrence
                co_terms[word] += 1

                # Track GBV Lexicon co-occurrence
                if word in GBV_FRENCH_LEXICON:
                    gbv_counts[word] += 1

# results
print(f"Matches found: {matches}\n")

print("--- top 10 overall co-occurring terms ---")
for term, count in co_terms.most_common(10):
    print(f"  {term}: {count}")

print("\n--- top GBV lexicon words appearing near target ---")
if gbv_counts:
    for term, count in gbv_counts.most_common:
        print(f"  {term}: {count}")
else:
    print("  No GBV lexicon terms found within the window size.")

print("\n--- sample context windows ---")
for ex in examples:
    print(f"- ... {ex} ...")

Matches found: 0

--- top 10 overall co-occurring terms ---

--- top GBV lexicon words appearing near target ---
  No GBV lexicon terms found within the window size.

--- sample context windows ---
